[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/101.%20The%20Supply%20Chain%20Finance%20%26%20Working%20Capital%20Optimization%20Problem/P101-Tier-2_executed.ipynb)

---


# 101. The Supply Chain Finance and Working Capital Optimization Problem

## Tier 2: A Priority-Based Online Heuristic

### Purpose and scope

This notebook illustrates a sequential, rule-based heuristic for handling payment requests when future requests and cash flows are not known at decision time. The algorithm receives requests over time, evaluates immediate payment, credit, and delayed-payment options, and records a decision using current liquidity, urgency, and request priority.

The inputs are an initial cash balance, a credit limit, entity-level request-generation profiles, and a simulated sequence of cash flows and payment requests. The outputs are the decisions and their estimated costs, daily liquidity and credit measures, diagnostic ratio values, plots, and scenario summaries.

### Model assumptions

- Decisions are made sequentially using information available on the current simulation day; future requests and cash flows are not supplied to the decision rule.
- Requests are ordered by priority and then due date. A request is evaluated once, when it is processed.
- A payment marked `pay_later` receives a planned payment date and an estimated delay cost. This teaching simulation does not model the later settlement transaction or a full accounts-payable ledger.
- Daily cash flows are simulated and added to the cash balance. Positive cash flows may also repay part of outstanding credit.
- The minimum balance is a reserve threshold for immediate-payment decisions; stochastic cash flows can move the balance below it.
        
The model does not represent taxes, transaction-level settlement, default, interest compounding, multiple currencies, or a complete network of inter-entity cash transfers. The stochastic simulation is illustrative rather than a calibrated forecast.

### How the heuristic works

1. Update cash and credit state using the day's cash flow.
2. Put newly arrived requests into a priority-ordered queue.
3. For each request selected for processing, estimate a diagnostic urgency-and-liquidity score and the costs of feasible payment options.
4. Use the score and request priority to choose an option, then record the decision.
5. Summarize and visualize the resulting decisions and resource use.

The value `1.26` is used as a heuristic decision threshold in this example. The score implemented here is **not** a formally established competitive ratio, and the threshold does not provide a competitive-performance guarantee. A formal competitive ratio would require a precise objective, a defined online policy, and comparison against an offline optimum over the same instances.

### Mathematical description

Let $r$ be a payment request with amount $a_r$, due date $d_r$, and priority $p_r$. At day $t$, let $B_t$ denote cash, $L$ the credit limit, and $U_t$ the amount of credit currently used. The available credit is $L-U_t$. The heuristic computes the diagnostic score

$$
q_{r,t} = \min\left\{3,\; \frac{1}{\max(1,d_r-t)}\left[2-\ell_t-c_t\right]\right\},
$$

where $\ell_t=\min(1,\max(0,B_t/B_0))$ is normalized cash liquidity, $c_t=\min(1,\max(0,(L-U_t)/L))$ is normalized available credit, and $B_0$ is the initial balance. The code implements this score in `calculate_competitive_ratio_bound`. It is a bounded heuristic indicator, not a theorem about worst-case performance.

The option-cost estimates are

$$
C_{\mathrm{now}}(r)=0.001a_r,\qquad
C_{\mathrm{credit}}(r,t)=0.05a_r\frac{d_r-t}{365},\qquad
C_{\mathrm{later}}(r,t)=0.02a_r\frac{\min(5,d_r-t)}{365}.
$$

An option is considered only when its corresponding feasibility condition holds: immediate payment must preserve the reserve threshold, credit use must fit within available credit, and delayed payment requires a future due date. The code evaluates these options in `evaluate_payment_options`; `make_payment_decision` applies the priority and score rules. These cost rates are illustrative assumptions, not market quotes.

This is a heuristic decision process rather than a dynamic-programming recurrence or an optimization model with a proven optimum. Requests are sorted and processed sequentially, and the simulation state grows with the number of days and requests. Sorting $n$ queued requests takes $O(n\log n)$ time; processing each request requires a constant number of option calculations. The random simulation is not a proof of a worst-case guarantee.

### What to examine

- Which payment options are selected and how often requests are rejected.
- How cash balances and credit usage change during the simulated period.
- How the diagnostic score behaves relative to its heuristic threshold.
- How the same heuristic responds to the scenarios defined later in the notebook.

The example uses a 30-day simulation, five entity profiles, an initial balance of $2,000,000, and a $4,500,000 credit limit. Requests and cash flows are randomly generated; because no random seed is set, a fresh execution can produce different results.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import copy
import random

import matplotlib.pyplot as plt
import numpy as np

print("Libraries imported for the online supply chain finance simulation.")

In [ ]:
@dataclass
class PaymentRequest:
    """A payment request received by the online decision process."""

    request_id: str
    from_entity: str
    to_entity: str
    amount: float
    due_date: int
    priority: int
    arrival_time: int


@dataclass
class WorkingCapitalBuffer:
    """Cash reserve and credit-facility state used by the heuristic."""

    current_balance: float
    minimum_balance: float
    target_balance: float
    credit_facility_available: float
    credit_facility_used: float


@dataclass
class OnlineSCFDecision:
    """The recorded choice and diagnostic score for one processed request."""

    request_id: str
    decision: str
    payment_date: int
    credit_used: float
    cost_impact: float
    decision_time: int
    competitive_ratio: float


@dataclass
class OnlineSCFState:
    """Mutable state carried forward through the daily simulation."""

    current_day: int
    buffer: WorkingCapitalBuffer
    pending_requests: List[PaymentRequest] = field(default_factory=list)
    decisions: List[OnlineSCFDecision] = field(default_factory=list)
    cash_flows: List[float] = field(default_factory=list)
    competitive_ratio: float = 0.0


print("Data structures for payment requests, decisions, and simulation state are ready.")

In [ ]:
class DynamicOnlineSCFAlgorithm:
    """Priority-based heuristic for sequential supply chain finance decisions."""

    def __init__(
        self,
        initial_balance: float,
        credit_limit: float,
        min_balance_ratio: float = 0.2,
    ):
        if initial_balance <= 0:
            raise ValueError("initial_balance must be positive.")
        if credit_limit <= 0:
            raise ValueError("credit_limit must be positive.")
        if not 0 <= min_balance_ratio <= 1:
            raise ValueError("min_balance_ratio must be between 0 and 1.")

        self.initial_balance = initial_balance
        self.credit_limit = credit_limit
        self.min_balance_ratio = min_balance_ratio
        self.state = OnlineSCFState(
            current_day=0,
            buffer=WorkingCapitalBuffer(
                current_balance=initial_balance,
                minimum_balance=initial_balance * min_balance_ratio,
                target_balance=initial_balance * 0.5,
                credit_facility_available=credit_limit,
                credit_facility_used=0.0,
            ),
        )

        # These values define the example heuristic; they are not proven bounds.
        self.competitive_ratio_target = 1.26
        self.buffer_safety_factor = 1.5
        self.priority_weights = {1: 1.0, 2: 0.8, 3: 0.6, 4: 0.4, 5: 0.2}

    def calculate_competitive_ratio_bound(
        self, request: PaymentRequest
    ) -> float:
        """Return the bounded urgency-and-liquidity diagnostic score."""
        days_to_due = max(1, request.due_date - self.state.current_day)
        urgency_factor = 1.0 / days_to_due

        liquidity_ratio = np.clip(
            self.state.buffer.current_balance / self.initial_balance,
            0.0,
            1.0,
        )
        available_credit = (
            self.state.buffer.credit_facility_available
            - self.state.buffer.credit_facility_used
        )
        credit_ratio = np.clip(available_credit / self.credit_limit, 0.0, 1.0)

        score = urgency_factor * (2.0 - liquidity_ratio - credit_ratio)
        return float(np.clip(score, 0.0, 3.0))

    def evaluate_payment_options(
        self, request: PaymentRequest
    ) -> Dict[str, float]:
        """Estimate costs for payment choices that satisfy their simple limits."""
        options = {}
        buffer = self.state.buffer
        available_credit = (
            buffer.credit_facility_available - buffer.credit_facility_used
        )
        days_to_due = max(0, request.due_date - self.state.current_day)

        # Preserving the reserve makes the minimum balance operational in the rule.
        if (
            buffer.current_balance - request.amount
            >= buffer.minimum_balance
        ):
            options["pay_now"] = request.amount * 0.001

        if available_credit >= request.amount:
            options["use_credit"] = (
                request.amount * 0.05 * days_to_due / 365
            )

        if request.due_date > self.state.current_day:
            delay_days = min(5, days_to_due)
            options["pay_later"] = (
                request.amount * 0.02 * delay_days / 365
            )

        return options

    def make_payment_decision(
        self, request: PaymentRequest
    ) -> OnlineSCFDecision:
        """Apply the heuristic rules and update cash or credit state."""
        score = self.calculate_competitive_ratio_bound(request)
        options = self.evaluate_payment_options(request)

        decision = "reject"
        payment_date = self.state.current_day
        credit_used = 0.0
        cost_impact = 0.0

        if options and score <= self.competitive_ratio_target:
            if "pay_now" in options:
                decision = "pay_now"
                cost_impact = options["pay_now"]
                self.state.buffer.current_balance -= request.amount
            elif "use_credit" in options:
                decision = "use_credit"
                credit_used = request.amount
                cost_impact = options["use_credit"]
                self.state.buffer.credit_facility_used += credit_used
            elif "pay_later" in options:
                decision = "pay_later"
                payment_date = min(self.state.current_day + 3, request.due_date)
                cost_impact = options["pay_later"]
        elif options:
            if request.priority <= 2 and "pay_now" in options:
                decision = "pay_now"
                cost_impact = options["pay_now"]
                self.state.buffer.current_balance -= request.amount
            elif request.priority <= 3 and "use_credit" in options:
                decision = "use_credit"
                credit_used = request.amount
                cost_impact = options["use_credit"]
                self.state.buffer.credit_facility_used += credit_used
            elif "pay_later" in options:
                decision = "pay_later"
                payment_date = min(self.state.current_day + 3, request.due_date)
                cost_impact = options["pay_later"]

        return OnlineSCFDecision(
            request_id=request.request_id,
            decision=decision,
            payment_date=payment_date,
            credit_used=credit_used,
            cost_impact=cost_impact,
            decision_time=self.state.current_day,
            competitive_ratio=score,
        )

    def process_daily_cash_flows(
        self, day: int, cash_flow_volatility: float = 400000
    ) -> float:
        """Generate and apply one bounded daily cash flow."""
        if cash_flow_volatility < 0:
            raise ValueError("cash_flow_volatility must be non-negative.")

        daily_cash_flow = np.random.normal(350000, cash_flow_volatility)
        daily_cash_flow = float(np.clip(daily_cash_flow, -500000, 1200000))
        self.state.buffer.current_balance += daily_cash_flow
        self.state.cash_flows.append(daily_cash_flow)

        # Positive inflows repay a portion of outstanding credit in this model.
        if daily_cash_flow > 0 and self.state.buffer.credit_facility_used > 0:
            repayment = min(
                daily_cash_flow * 0.3,
                self.state.buffer.credit_facility_used,
            )
            self.state.buffer.credit_facility_used -= repayment
            self.state.buffer.current_balance -= repayment

        return daily_cash_flow

    def simulate_day(
        self,
        day: int,
        new_requests: List[PaymentRequest],
        cash_flow_volatility: float = 400000,
    ) -> None:
        """Advance the system by one day and process up to ten queued requests."""
        self.state.current_day = day
        self.process_daily_cash_flows(day, cash_flow_volatility)
        self.state.pending_requests.extend(new_requests)
        self.state.pending_requests.sort(
            key=lambda request: (request.priority, request.due_date)
        )

        # The daily capacity limit leaves excess requests in the queue for later days.
        requests_to_process = self.state.pending_requests[:10]
        for request in requests_to_process:
            decision = self.make_payment_decision(request)
            self.state.decisions.append(decision)
        self.state.pending_requests = self.state.pending_requests[
            len(requests_to_process):
        ]

        daily_scores = [
            decision.competitive_ratio
            for decision in self.state.decisions
            if decision.decision_time == day
        ]
        if daily_scores:
            self.state.competitive_ratio = float(np.mean(daily_scores))


print("The sequential payment heuristic is defined.")

In [ ]:
def create_simulation_environment():
    """Create the example algorithm and five entity request profiles."""
    algorithm = DynamicOnlineSCFAlgorithm(
        initial_balance=2000000,
        credit_limit=4500000,
        min_balance_ratio=0.2,
    )

    entities = {
        "Tier4_Supplier": {
            "payment_frequency": 0.3,
            "avg_amount": 50000,
            "priority_range": (3, 5),
        },
        "Tier3_Manufacturer": {
            "payment_frequency": 0.4,
            "avg_amount": 100000,
            "priority_range": (2, 4),
        },
        "Tier2_SubAssembly": {
            "payment_frequency": 0.5,
            "avg_amount": 200000,
            "priority_range": (2, 3),
        },
        "Tier1_Assembly": {
            "payment_frequency": 0.6,
            "avg_amount": 300000,
            "priority_range": (1, 3),
        },
        "OEM": {
            "payment_frequency": 0.2,
            "avg_amount": 500000,
            "priority_range": (1, 2),
        },
    }
    return algorithm, entities


def generate_daily_requests(
    day: int,
    entities: Dict,
    request_frequency_multiplier: float = 1.0,
) -> List[PaymentRequest]:
    """Generate requests using the supplied entity profiles."""
    if request_frequency_multiplier < 0:
        raise ValueError("request_frequency_multiplier must be non-negative.")

    requests = []
    for entity_name, profile in entities.items():
        probability = min(
            1.0,
            profile["payment_frequency"] * request_frequency_multiplier,
        )
        if random.random() < probability:
            amount = np.random.normal(
                profile["avg_amount"],
                profile["avg_amount"] * 0.3,
            )
            amount = max(10000, float(amount))
            priority = random.randint(*profile["priority_range"])
            due_date = day + random.randint(1, 30)
            requests.append(
                PaymentRequest(
                    request_id=f"R{day}_{entity_name}",
                    from_entity=entity_name,
                    to_entity="Next_Tier",
                    amount=amount,
                    due_date=due_date,
                    priority=priority,
                    arrival_time=day,
                )
            )
    return requests


algorithm, entities = create_simulation_environment()
print(f"Simulation created with {len(entities)} entity profiles.")
print(f"Initial cash balance: ${algorithm.initial_balance:,.0f}")
print(f"Credit limit: ${algorithm.credit_limit:,.0f}")

In [ ]:
def run_simulation(
    algorithm: DynamicOnlineSCFAlgorithm,
    entities: Dict,
    days: int = 30,
    cash_flow_volatility: float = 400000,
    request_frequency_multiplier: float = 1.0,
) -> Dict:
    """Run the daily online process and collect its observable metrics."""
    if days <= 0:
        raise ValueError("days must be positive.")

    results = {
        "daily_decisions": [],
        "daily_cash_flows": [],
        "daily_balances": [],
        "daily_credit_usage": [],
        "daily_competitive_ratios": [],
        "total_requests": 0,
        "processed_requests": 0,
        "rejected_requests": 0,
        "total_cost": 0.0,
    }

    for day in range(days):
        daily_requests = generate_daily_requests(
            day,
            entities,
            request_frequency_multiplier=request_frequency_multiplier,
        )
        results["total_requests"] += len(daily_requests)

        decisions_before = len(algorithm.state.decisions)
        algorithm.simulate_day(
            day,
            daily_requests,
            cash_flow_volatility=cash_flow_volatility,
        )
        day_decisions = algorithm.state.decisions[decisions_before:]

        results["processed_requests"] += sum(
            decision.decision != "reject" for decision in day_decisions
        )
        results["rejected_requests"] += sum(
            decision.decision == "reject" for decision in day_decisions
        )
        results["daily_decisions"].append(len(day_decisions))
        results["daily_cash_flows"].append(algorithm.state.cash_flows[-1])
        results["daily_balances"].append(
            algorithm.state.buffer.current_balance
        )
        results["daily_credit_usage"].append(
            algorithm.state.buffer.credit_facility_used
        )
        results["daily_competitive_ratios"].append(
            algorithm.state.competitive_ratio
        )
        results["total_cost"] += sum(
            decision.cost_impact for decision in day_decisions
        )

    return results


print("Running the 30-day simulation...")
simulation_results = run_simulation(algorithm, entities, days=30)

total_requests = simulation_results["total_requests"]
success_rate = (
    100 * simulation_results["processed_requests"] / total_requests
    if total_requests
    else 0.0
)
average_score = float(
    np.mean(simulation_results["daily_competitive_ratios"])
)
average_credit_usage = float(
    np.mean(simulation_results["daily_credit_usage"])
)
credit_utilization = 100 * average_credit_usage / algorithm.credit_limit
final_balance = simulation_results["daily_balances"][-1]

print("\nSimulation summary")
print(f"Requests received: {total_requests}")
print(f"Requests accepted: {simulation_results['processed_requests']}")
print(f"Requests rejected: {simulation_results['rejected_requests']}")
print(f"Acceptance rate: {success_rate:.1f}%")
print(f"Estimated decision costs: ${simulation_results['total_cost']:,.2f}")
print(f"Average diagnostic score: {average_score:.2f}")
print(f"Average credit utilization: {credit_utilization:.1f}%")
print(f"Final cash balance: ${final_balance:,.0f}")

In [ ]:
def create_online_algorithm_visualizations(
    simulation_results: Dict,
    algorithm: DynamicOnlineSCFAlgorithm,
):
    """Plot daily liquidity, decisions, diagnostic scores, and credit use."""
    days = np.arange(1, len(simulation_results["daily_cash_flows"]) + 1)
    if len(days) == 0:
        raise ValueError("The simulation results contain no daily observations.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(
        "Online Supply Chain Finance Heuristic: Simulation Results",
        fontsize=15,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    balance_axis = ax1.twinx()
    cash_bars = ax1.bar(
        days,
        simulation_results["daily_cash_flows"],
        alpha=0.6,
        color="lightblue",
        label="Daily cash flow",
    )
    balance_line, = balance_axis.plot(
        days,
        simulation_results["daily_balances"],
        color="darkred",
        linewidth=2,
        label="Cash balance",
    )
    ax1.set_xlabel("Day")
    ax1.set_ylabel("Daily cash flow ($)")
    balance_axis.set_ylabel("Cash balance ($)")
    ax1.set_title("Daily Cash Flow and Cash Balance")
    ax1.legend([cash_bars, balance_line], ["Daily cash flow", "Cash balance"])
    ax1.grid(True, alpha=0.25)

    decision_counts = {
        "pay_now": 0,
        "pay_later": 0,
        "use_credit": 0,
        "reject": 0,
    }
    for decision in algorithm.state.decisions:
        decision_counts[decision.decision] += 1
    ax2 = axes[0, 1]
    if sum(decision_counts.values()) > 0:
        ax2.pie(
            list(decision_counts.values()),
            labels=list(decision_counts.keys()),
            colors=["lightgreen", "lightblue", "orange", "lightcoral"],
            autopct="%1.1f%%",
            startangle=90,
        )
    else:
        ax2.text(0.5, 0.5, "No payment decisions", ha="center", va="center")
    ax2.set_title("Recorded Payment Decisions")

    ax3 = axes[1, 0]
    score_line, = ax3.plot(
        days,
        simulation_results["daily_competitive_ratios"],
        color="navy",
        linewidth=2,
        marker="o",
        markersize=4,
        label="Daily mean diagnostic score",
    )
    threshold_line = ax3.axhline(
        y=algorithm.competitive_ratio_target,
        color="red",
        linestyle="--",
        label=f"Heuristic threshold ({algorithm.competitive_ratio_target:.2f})",
    )
    ax3.set_xlabel("Day")
    ax3.set_ylabel("Diagnostic score (unitless)")
    ax3.set_title("Urgency-and-Liquidity Diagnostic")
    ax3.legend(handles=[score_line, threshold_line])
    ax3.grid(True, alpha=0.25)
    ax3.set_ylim(0, 3)

    ax4 = axes[1, 1]
    credit_usage_pct = [
        100 * usage / algorithm.credit_limit
        for usage in simulation_results["daily_credit_usage"]
    ]
    ax4.fill_between(days, 0, credit_usage_pct, alpha=0.3, color="orange")
    ax4.plot(days, credit_usage_pct, color="darkorange", linewidth=2)
    ax4.set_xlabel("Day")
    ax4.set_ylabel("Credit used (% of limit)")
    ax4.set_title("Credit-Facility Utilization")
    ax4.set_ylim(0, 100)
    ax4.grid(True, alpha=0.25)

    fig.tight_layout()
    plt.show()
    return fig


visualization_fig = create_online_algorithm_visualizations(
    simulation_results,
    algorithm,
)

In [ ]:
def analyze_algorithm_performance(
    simulation_results: Dict,
    algorithm: DynamicOnlineSCFAlgorithm,
) -> Dict[str, float]:
    """Summarize observed costs, resource use, and request outcomes."""
    decisions = algorithm.state.decisions
    total_decisions = len(decisions)
    accepted_decisions = sum(
        decision.decision != "reject" for decision in decisions
    )
    immediate_or_credit = sum(
        decision.decision in {"pay_now", "use_credit"}
        for decision in decisions
    )
    average_score = (
        float(np.mean(simulation_results["daily_competitive_ratios"]))
        if simulation_results["daily_competitive_ratios"]
        else 0.0
    )
    average_credit_usage = (
        float(np.mean(simulation_results["daily_credit_usage"]))
        if simulation_results["daily_credit_usage"]
        else 0.0
    )
    total_requests = simulation_results["total_requests"]

    return {
        "estimated_decision_cost": simulation_results["total_cost"],
        "average_credit_utilization_pct": (
            100 * average_credit_usage / algorithm.credit_limit
        ),
        "average_diagnostic_score": average_score,
        "accepted_request_pct": (
            100 * accepted_decisions / total_requests
            if total_requests
            else 0.0
        ),
        "immediate_or_credit_pct": (
            100 * immediate_or_credit / total_decisions
            if total_decisions
            else 0.0
        ),
        "requests_left_pending": len(algorithm.state.pending_requests),
    }


performance = analyze_algorithm_performance(simulation_results, algorithm)
print("Observed performance measures")
print(f"Estimated decision costs: ${performance['estimated_decision_cost']:,.2f}")
print(
    "Average credit utilization: "
    f"{performance['average_credit_utilization_pct']:.1f}%"
)
print(f"Average diagnostic score: {performance['average_diagnostic_score']:.2f}")
print(f"Accepted requests: {performance['accepted_request_pct']:.1f}%")
print(
    "Immediate-payment or credit decisions: "
    f"{performance['immediate_or_credit_pct']:.1f}%"
)
print(f"Requests still queued: {performance['requests_left_pending']}")

In [ ]:
def perform_what_if_analysis(
    entities: Dict,
    days: int = 15,
) -> Dict[str, Dict[str, float]]:
    """Run the same heuristic under the stated illustrative scenarios."""
    scenarios = {
        "High Volatility": {
            "cash_flow_volatility": 800000,
            "request_frequency_multiplier": 1.0,
            "initial_balance": 2000000,
            "credit_limit": 4500000,
            "priority_shift": 0,
        },
        "Low Liquidity": {
            "cash_flow_volatility": 400000,
            "request_frequency_multiplier": 1.0,
            "initial_balance": 1000000,
            "credit_limit": 2000000,
            "priority_shift": 0,
        },
        "High Priority Load": {
            "cash_flow_volatility": 400000,
            "request_frequency_multiplier": 1.0,
            "initial_balance": 2000000,
            "credit_limit": 4500000,
            "priority_shift": 1,
        },
        "Extended Credit": {
            "cash_flow_volatility": 400000,
            "request_frequency_multiplier": 1.0,
            "initial_balance": 2000000,
            "credit_limit": 8000000,
            "priority_shift": 0,
        },
    }

    scenario_results = {}
    for scenario_name, parameters in scenarios.items():
        test_algorithm = DynamicOnlineSCFAlgorithm(
            initial_balance=parameters["initial_balance"],
            credit_limit=parameters["credit_limit"],
        )
        test_entities = copy.deepcopy(entities)

        if parameters["priority_shift"]:
            shift = parameters["priority_shift"]
            for profile in test_entities.values():
                lower, upper = profile["priority_range"]
                profile["priority_range"] = (
                    max(1, lower - shift),
                    max(1, upper - shift),
                )

        results = run_simulation(
            test_algorithm,
            test_entities,
            days=days,
            cash_flow_volatility=parameters["cash_flow_volatility"],
            request_frequency_multiplier=parameters[
                "request_frequency_multiplier"
            ],
        )
        metrics = analyze_algorithm_performance(results, test_algorithm)
        metrics["total_requests"] = results["total_requests"]
        metrics["rejected_requests"] = results["rejected_requests"]
        metrics["estimated_decision_cost"] = results["total_cost"]
        scenario_results[scenario_name] = metrics

    return scenario_results


what_if_results = perform_what_if_analysis(entities)
print("What-if scenario summaries")
for scenario_name, metrics in what_if_results.items():
    print(f"\n{scenario_name}")
    print(f"  Requests received: {metrics['total_requests']}")
    print(f"  Requests rejected: {metrics['rejected_requests']}")
    print(f"  Estimated decision costs: ${metrics['estimated_decision_cost']:,.2f}")
    print(
        "  Average diagnostic score: "
        f"{metrics['average_diagnostic_score']:.2f}"
    )
    print(
        "  Average credit utilization: "
        f"{metrics['average_credit_utilization_pct']:.1f}%"
    )

scenario_names = list(what_if_results)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Illustrative What-If Scenario Comparison", fontsize=15)

plot_specs = [
    ("estimated_decision_cost", "Estimated decision cost ($)"),
    ("accepted_request_pct", "Accepted requests (%)"),
    ("average_diagnostic_score", "Average diagnostic score"),
    ("average_credit_utilization_pct", "Average credit utilization (%)"),
]
for axis, (metric, label) in zip(axes.flat, plot_specs):
    values = [what_if_results[name][metric] for name in scenario_names]
    axis.bar(scenario_names, values, color="steelblue", alpha=0.75)
    axis.set_ylabel(label)
    axis.set_title(label)
    axis.tick_params(axis="x", rotation=35)
    axis.grid(True, axis="y", alpha=0.25)

fig.tight_layout()
plt.show()

## Interpretation and limitations

The simulation illustrates how a priority-based online heuristic can turn currently available information into sequential payment choices. Its reported costs are the option-cost estimates attached to recorded decisions; they are not a complete measure of financing expense. The credit-utilization measure describes the simulated credit state, while the diagnostic score summarizes the particular urgency-and-liquidity expression defined above.

The scenario analysis applies the same decision rules to different cash-flow volatility, liquidity, priority, or credit settings. Since requests and cash flows are random and no seed is fixed, differences between scenario runs include both parameter effects and random variation. The results are descriptive observations from these simulations, not guarantees or statistically controlled comparisons.

### Strengths and limitations of the online approach

**Strengths**

- Decisions use current information and do not require a forecast of the full future request sequence.
- The rule explicitly considers cash availability, a reserve threshold, credit capacity, due dates, and request priority.
- The state and decision steps are visible, making the heuristic straightforward to trace in a small simulation.

**Limitations**

- A heuristic decision is not a proof of global optimality or a formal competitive-ratio guarantee.
- The cost rates and entity profiles are illustrative, and performance can be sensitive to those assumptions.
- Delayed-payment choices are recorded as plans, but later settlement is outside this simulation's accounting scope.
- Cash-flow randomness, a simplified credit repayment rule, and the absence of a complete payment ledger limit real-world interpretation.

### When this tier is useful

This tier is useful for studying the logic of sequential decisions when information arrives over time. It can help readers identify which state variables and rules influence a payment choice, and why an online heuristic may behave differently from a planning model with complete future information. Applying such a rule operationally would require calibrated data, explicit settlement and service-level constraints, and validation appropriate to the intended use.